# Check agreement between independent annotators

Before learning from labels, ask whether independent annotators make
similar decisions on the same posts. This notebook compares decisions,
not the wording of their evidence quotes. It uses the shared 31-tag codebook.

**Reading route:** a small example → choose annotation sheets → align by
ID → distinguish missing from empty → calculate agreement → read reports.

Use annotations made independently, before discussion or adjudication. Outputs use Annotator A/B/C aliases.
No real scores are included in this sharing copy. Run the notebook with
your workbook to obtain the results for that workbook.

## 1. Start with a tiny example

For four fictional posts, two annotators decide whether anxiety is
present. They agree on three decisions, so percent agreement is 75%.
However, some agreement is expected from their category frequencies.
Cohen's kappa adjusts for that chance baseline.

$$\kappa = \frac{P_o-P_e}{1-P_e}$$

Here, observed agreement is 0.75. Annotator A selects anxiety for half
the posts; B selects it for three quarters. Chance agreement is
`(0.5 × 0.75) + (0.5 × 0.25) = 0.5`. Kappa is therefore 0.5.

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
toy_a = np.array([1, 1, 0, 0])
toy_b = np.array([1, 1, 1, 0])
print('Fictional percent agreement:', 100 * np.mean(toy_a == toy_b))
print('Fictional Cohen kappa:', cohen_kappa_score(toy_a, toy_b))

## Connect to the shared project files

The `src/` folder is the shared toolbox. It holds the codebook and functions used
by multiple notebooks. The next cell finds it from the project root or `notebooks/`.
Keep the complete repository together. On Kaggle or Colab, extract or attach the
complete repository and set `PROJECT_ROOT` to its location when necessary.

Install the repository requirements before running the notebook. The setup cell
prints the root and label count so you can check the selected files. Data and
generated results remain in the ignored project folders.

In [ ]:
from pathlib import Path
import os
import sys

# Leave None when running from the repository root or notebooks/.
# On Kaggle/Colab, set this to the extracted repository directory when needed.
PROJECT_ROOT = os.environ.get("MINDMINER_PROJECT_ROOT")
if PROJECT_ROOT:
    REPO_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    REPO_ROOT = next((p for p in [here, *here.parents]
                      if (p / "src/mindminer_codebook.py").is_file()), None)
if REPO_ROOT is None or not (REPO_ROOT / "src/mindminer_codebook.py").is_file():
    raise FileNotFoundError("Extract the complete MindMiner repository and set PROJECT_ROOT to its folder.")
sys.path.insert(0, str(REPO_ROOT / "src"))
RUN_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else REPO_ROOT
from mindminer_codebook import CODEBOOK_VERSION, CODEBOOK_SHA256, CODEBOOK, FIELDS, TAGS, LABELS
print("Project root:", REPO_ROOT)
print("Codebook:", CODEBOOK_VERSION, "| tags:", len(LABELS))

In [ ]:
import pandas as pd
from IPython.display import display
from mindminer_agreement import (
    load_annotations, available_single_fields, calculate_agreement,
    compare_models, save_agreement, save_model_comparison,
)
from mindminer_utils import file_hash, save_json

## 2. Choose the workbook and sheets

An Excel workbook contains one sheet per annotation set. Each selected
sheet needs `post_id` and the four tag columns: `stressors`,
`relationships`, `emotions`, and `responses`. If all selected sheets
also contain TBe/PBu metadata (`belong`, `burden`, `group`), those fields
can be compared separately. Copied metadata does not measure independent
annotation agreement.

Map a public rater name to its actual sheet name below. Change the right
side if your workbook uses different names. Unselected master, consensus,
or adjudicated sheets are not included automatically.

IDs align posts even when the sheets have different row orders. If IDs
repeat across splits, set `ID_COLUMNS = ['split', 'post_id']` and include
both columns in every sheet.

In [ ]:
FILE = Path(os.environ.get("MINDMINER_AGREEMENT_FILE", str(REPO_ROOT / "data/annotations/annotation_workbook.xlsx")))
# Left = public output alias. Right = existing workbook sheet name.
SHEET_MAP = {"Annotator A": "Pial_step1", "Annotator B": "Farhan_step1", "Annotator C": "Nasif_step1"}
ID_COLUMNS = ["post_id"]
REQUESTED_SINGLE_FIELDS = ["belong", "burden", "group"]
BLANK_MULTILABEL_IS_EMPTY = True
OUTPUT = RUN_ROOT / "results/agreement/human_agreement"

## 3. Missing annotation is different from an empty tag set

| Cell content | Meaning in this notebook |
| --- | --- |
| `anxiety; sadness` | Both tags were selected; commas also work |
| `not stated`, `none`, `no labels`, `[]`, `{}` | Completed annotation with no selected tags |
| `missing`, `unannotated`, `not reviewed`, `n/a`, `na` | The annotation is unavailable and must be excluded |
| Blank | Empty if `BLANK_MULTILABEL_IS_EMPTY=True`; missing if False |

Choose the blank policy to match your annotation process. Do not treat
an unfinished row as a completed “no tags” decision. Missing rows on a
sheet remain unavailable after ID alignment. Single-label blanks are
always missing. Unknown tags and mixed markers raise a clear error.

The codebook defines all 31 decisions in advance. A tag absent from every
annotation remains in the output, with zero support and undefined kappa
where appropriate. `not stated` is not an additional prediction tag.

In [ ]:
frames, alignment = load_annotations(FILE, SHEET_MAP, ID_COLUMNS)
single_fields = available_single_fields(frames, REQUESTED_SINGLE_FIELDS)
display(alignment)
print('Additional categorical fields:', single_fields)
print('Blank tag cells mean:', 'empty set' if BLANK_MULTILABEL_IS_EMPTY else 'missing rating')

## 4. Calculate symmetric agreement

For each tag, turn its presence into a yes/no decision. Cohen's kappa
compares two raters after excluding unavailable ratings for that pair.
The sample size and positive counts are saved beside every score.

For a whole field, **exact-set agreement** asks whether both selected
exactly the same tags. **Jaccard** measures partial overlap. If one set
is `{anxiety, sadness}` and the other is `{anxiety}`, their Jaccard is
`1 / 2 = 0.5`, even though exact-set agreement is zero for that post.

Two empty sets receive Jaccard 1. An additional active-only Jaccard
excludes posts where both sets are empty, with the retained count shown.
All these pairwise agreement quantities are symmetric: swapping the
raters does not change their values.

In [ ]:
tables = calculate_agreement(
    frames, single_fields=single_fields,
    blank_is_empty=BLANK_MULTILABEL_IS_EMPTY,
    include_fleiss=True,
)
display(tables['pairwise_kappa'].round(4))
display(tables['multilabel_summary'].round(4))

### What does Fleiss' kappa add?

Fleiss' kappa summarizes categorical agreement across the three
selected annotators. The calculation uses posts with a usable rating
from every selected person for that field/tag. Its chance baseline
uses pooled category frequencies. Pairwise Cohen scores may therefore
use more rows than the all-rater Fleiss score.

A score of 1 represents perfect agreement when kappa is defined;
0 is its chance baseline, and negative values are below that baseline.
Constant identical ratings have undefined kappa, saved as `NA`, rather
than being assigned 1. High percent agreement can come from many
shared absences, so inspect positive counts and label support too.

In [ ]:
display(tables['fleiss_kappa'].round(4))

## Save results and interpret them

| Output | What to read |
| --- | --- |
| `pairwise_kappa.csv` | Each pair's per-tag kappa, agreement, support, and usable count |
| `multilabel_summary.csv` | Exact-set agreement, all-post and active-only Jaccard |
| `alignment_audit.csv` | Rater row counts and shared/unshared ID counts |
| `run_settings.json` | Input hash, sheet mapping, codebook identity, and blank policy |

The human notebook also saves `fleiss_kappa.csv`. The model notebook
saves directional field/per-label tables and classification reports
under one subfolder per direction. These generated results remain under
`results/agreement/`; the original workbook is unchanged.

The mean of defined per-label kappas is a descriptive average, not a
pooled kappa. Scores depend on which labels are present and how common
they are. Report counts and the missing/empty conventions with results.

In [ ]:
save_agreement(tables, OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
alignment.to_csv(OUTPUT / 'alignment_audit.csv', index=False)
save_json(OUTPUT / 'run_settings.json', {
    'input_sha256': file_hash(FILE), 'sheet_map': SHEET_MAP, 'id_columns': ID_COLUMNS,
    'codebook_version': CODEBOOK_VERSION, 'codebook_sha256': CODEBOOK_SHA256,
    'target_labels': LABELS, 'single_fields': single_fields,
    'blank_multilabel_is_empty': BLANK_MULTILABEL_IS_EMPTY,
})

In [ ]:
print('Saved results:', OUTPUT)

## Explain this analysis in your presentation

Start with the annotation sets and shared sample. State whether they
were independent, which fields were rated, and how missing values were
handled. Then show percent agreement, per-label kappa, and set overlap
with sample sizes. Explain uncertain or rare tags using actual results.

These notebooks do not calculate confidence intervals or establish
semantic correctness from agreement alone. Keep executed outputs cleared
before sharing notebooks that have displayed identifiers or post content.

Definitions: [Cohen's kappa](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html),
[Fleiss' kappa](https://www.statsmodels.org/stable/generated/statsmodels.stats.inter_rater.fleiss_kappa.html).